In [1]:
# ============================================================
# MatricMath Intelligence
# Notebook 01: Exam Document Inventory
# ============================================================
# Purpose:
# Establish the document control register before any bulk download.
# Output:
# data/metadata/exam_document_register.csv
# ============================================================

print("Notebook 01 – Exam Document Inventory")

Notebook 01 – Exam Document Inventory


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
META_DIR = DATA_DIR / "metadata"

EXAMS_DIR = RAW_DIR / "exams"
MEMOS_DIR = RAW_DIR / "memos"
DIAG_DIR = RAW_DIR / "diagnostic_reports"
CAPS_DIR = RAW_DIR / "caps"

for directory in [EXAMS_DIR, MEMOS_DIR, DIAG_DIR, CAPS_DIR, META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Folder structure ready")
print(f"Register path: {META_DIR / 'exam_document_register.csv'}")

Folder structure ready
Register path: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv


## Source Hierarchy

### Tier 1 — Official primary
- DBE examination papers
- DBE marking guidelines
- DBE diagnostic reports
- DBE CAPS documents
- DBE examination guidelines
- Umalusi reports

### Tier 2 — Official secondary
- Provincial education repositories

### Tier 3 — Backfill only
- Trusted educational archives, used only when Tier 1/2 are unavailable

Rule: Tier 1 first.

In [3]:
REGISTER_COLUMNS = [
    "document_id",
    "year",
    "exam_session",
    "paper",
    "document_type",
    "subject",
    "language",
    "source",
    "source_tier",
    "source_url",
    "file_name",
    "file_path",
    "curriculum_regime",
    "text_extractable",
    "ocr_required",
    "collection_status",
    "priority",
    "notes",
    "date_added",
]

print(f"Schema columns: {len(REGISTER_COLUMNS)}")

Schema columns: 19


In [4]:
def curriculum_regime(year: int) -> str:
    """
    Grade 12 / Matric Mathematics curriculum-regime classification.

    CAPS implementation:
    - 2012: Grade 10 starts CAPS
    - 2013: Grade 11 starts CAPS
    - 2014: Grade 12 / matric first fully assessed under CAPS
    """
    if year < 2012:
        return "pre-CAPS"
    if year in (2012, 2013):
        return "CAPS-transition"
    return "CAPS-Grade12"


def priority_for_year(year: int) -> str:
    """
    Collection priority.
    CAPS-Grade12 years are prioritised; recent years first.
    """
    if year >= 2023:
        return "high"
    if year >= 2018:
        return "medium"
    if year >= 2014:
        return "medium"   # CAPS-Grade12, still useful
    return "low"          # pre-CAPS / transition for later expansion


for year in [2010, 2012, 2013, 2014, 2018, 2024]:
    print(year, "→", curriculum_regime(year), "|", priority_for_year(year))

2010 → pre-CAPS | low
2012 → CAPS-transition | low
2013 → CAPS-transition | low
2014 → CAPS-Grade12 | medium
2018 → CAPS-Grade12 | medium
2024 → CAPS-Grade12 | high


In [5]:
records = []
now = datetime.now(timezone.utc).isoformat()

# Primary CAPS-Grade12 analytical window
YEARS = list(range(2014, 2026))

EXAM_SESSIONS = ["Nov", "May-June", "Supplementary", "Feb-March"]
PAPERS = ["P1", "P2"]
DOCUMENT_TYPES = ["exam", "memo"]

for year in YEARS:
    for session in EXAM_SESSIONS:
        for paper in PAPERS:
            for document_type in DOCUMENT_TYPES:
                document_id = (
                    f"{year}_{session}_{paper}_{document_type}_maths"
                    .lower()
                    .replace("-", "_")
                )
                records.append({
                    "document_id": document_id,
                    "year": year,
                    "exam_session": session,
                    "paper": paper,
                    "document_type": document_type,
                    "subject": "Mathematics",
                    "language": "Unknown",
                    "source": "DBE",
                    "source_tier": 1,
                    "source_url": "",
                    "file_name": "",
                    "file_path": "",
                    "curriculum_regime": curriculum_regime(year),
                    "text_extractable": "unknown",
                    "ocr_required": "unknown",
                    "collection_status": "queued",
                    "priority": priority_for_year(year),
                    "notes": "Inventory candidate; availability must be verified",
                    "date_added": now,
                })

# Diagnostic reports
for year in YEARS:
    records.append({
        "document_id": f"{year}_diagnostic_maths",
        "year": year,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "diagnostic",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": curriculum_regime(year),
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": priority_for_year(year),
        "notes": "DBE Mathematics diagnostic report target",
        "date_added": now,
    })

# Supporting curriculum documents
records.extend([
    {
        "document_id": "caps_mathematics_current",
        "year": 0,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "caps",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": "high",
        "notes": "CAPS Mathematics curriculum reference",
        "date_added": now,
    },
    {
        "document_id": "nsc_mathematics_exam_guideline",
        "year": 0,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "guideline",
        "subject": "Mathematics",
        "language": "Unknown",
        "source": "DBE",
        "source_tier": 1,
        "source_url": "",
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "queued",
        "priority": "high",
        "notes": "NSC Mathematics examination guideline",
        "date_added": now,
    },
])

register_df = pd.DataFrame(records, columns=REGISTER_COLUMNS)

print(f"Seed inventory rows: {len(register_df)}")
print("\nCurriculum regimes:")
print(register_df["curriculum_regime"].value_counts())
print("\nPriority distribution:")
print(register_df["priority"].value_counts())
display(register_df.head(15))

Seed inventory rows: 206

Curriculum regimes:
curriculum_regime
CAPS-Grade12    206
Name: count, dtype: int64

Priority distribution:
priority
medium    153
high       53
Name: count, dtype: int64


,document_id,year,exam_session,paper,document_type,subject,language,source,source_tier,source_url,file_name,file_path,curriculum_regime,text_extractable,ocr_required,collection_status,priority,notes,date_added
0,2014_nov_p1_exam_maths,2014,Nov,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
1,2014_nov_p1_memo_maths,2014,Nov,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
2,2014_nov_p2_exam_maths,2014,Nov,P2,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
3,2014_nov_p2_memo_maths,2014,Nov,P2,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
4,2014_may_june_p1_exam_maths,2014,May-June,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
5,2014_may_june_p1_memo_maths,2014,May-June,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
6,2014_may_june_p2_exam_maths,2014,May-June,P2,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
7,2014_may_june_p2_memo_maths,2014,May-June,P2,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
8,2014_supplementary_p1_exam_maths,2014,Supplementary,P1,exam,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00
9,2014_supplementary_p1_memo_maths,2014,Supplementary,P1,memo,Mathematics,Unknown,DBE,1,,,,CAPS-Grade12,unknown,unknown,queued,medium,Inventory candidate; availability must be veri...,2026-09-06T13:22:07.122120+00:00


In [6]:
assert list(register_df.columns) == REGISTER_COLUMNS
assert register_df["document_id"].is_unique
assert register_df["collection_status"].isin(
    ["queued", "downloaded", "verified", "missing"]
).all()
assert register_df["source_tier"].isin([1, 2, 3]).all()

# CAPS-Grade12 check for 2014+
mask_2014_plus = register_df["year"] >= 2014
assert (
    register_df.loc[mask_2014_plus, "curriculum_regime"] == "CAPS-Grade12"
).all()

print("Validation passed")

Validation passed


In [7]:
print("DOCUMENT INVENTORY SUMMARY")
print("=" * 60)
print(f"Total records        : {len(register_df)}")
print(f"Exam                 : {(register_df['document_type'] == 'exam').sum()}")
print(f"Memo                 : {(register_df['document_type'] == 'memo').sum()}")
print(f"Diagnostic           : {(register_df['document_type'] == 'diagnostic').sum()}")
print(f"Supporting           : {register_df['document_type'].isin(['caps','guideline']).sum()}")
print(f"High priority        : {(register_df['priority'] == 'high').sum()}")
print(f"Medium priority      : {(register_df['priority'] == 'medium').sum()}")
print(f"Low priority         : {(register_df['priority'] == 'low').sum()}")
print(f"CAPS-Grade12 rows    : {(register_df['curriculum_regime'] == 'CAPS-Grade12').sum()}")

DOCUMENT INVENTORY SUMMARY
Total records        : 206
Exam                 : 96
Memo                 : 96
Diagnostic           : 12
Supporting           : 2
High priority        : 53
Medium priority      : 153
Low priority         : 0
CAPS-Grade12 rows    : 206


In [8]:
exam_only = register_df[register_df["document_type"] == "exam"].copy()
session_matrix = pd.crosstab(exam_only["year"], exam_only["exam_session"])
display(session_matrix)

print("\nNote: matrix shows candidate inventory, not confirmed availability.")

exam_session,Feb-March,May-June,Nov,Supplementary
year,,,,
2014,2,2,2,2
2015,2,2,2,2
2016,2,2,2,2
2017,2,2,2,2
2018,2,2,2,2
2019,2,2,2,2
2020,2,2,2,2
2021,2,2,2,2
2022,2,2,2,2



Note: matrix shows candidate inventory, not confirmed availability.


In [9]:
register_path = META_DIR / "exam_document_register.csv"
register_df.to_csv(register_path, index=False)

print(f"Register saved → {register_path}")
print(f"Rows: {len(register_df)}")
print("NO BULK PDF DOWNLOAD PERFORMED")

Register saved → c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv
Rows: 206
NO BULK PDF DOWNLOAD PERFORMED


In [10]:
reloaded = pd.read_csv(register_path)

assert len(reloaded) == len(register_df)
assert list(reloaded.columns) == REGISTER_COLUMNS
assert reloaded["document_id"].is_unique

print("CSV reload verification passed")

CSV reload verification passed


In [11]:
# ============================================================
# LOAD CONTROL REGISTER
# ============================================================

register_path = META_DIR / "exam_document_register.csv"
register_df = pd.read_csv(register_path)

print(f"Loaded register: {register_path}")
print(f"Rows: {len(register_df)}")
print(register_df["collection_status"].value_counts())

Loaded register: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv
Rows: 206
collection_status
queued    206
Name: count, dtype: int64


In [12]:
# ============================================================
# OFFICIAL DBE AVAILABILITY AUDIT
# ============================================================
# "verified" means the document was confirmed on an official
# DBE landing page. It does NOT mean the PDF was downloaded.

VERIFIED_DOCUMENTS = [
    # 2025 November
    (2025, "Nov", "P1", "exam"),
    (2025, "Nov", "P1", "memo"),
    (2025, "Nov", "P2", "exam"),
    (2025, "Nov", "P2", "memo"),

    # 2024 November
    (2024, "Nov", "P1", "exam"),
    (2024, "Nov", "P1", "memo"),
    (2024, "Nov", "P2", "exam"),
    (2024, "Nov", "P2", "memo"),

    # 2023 November
    (2023, "Nov", "P1", "exam"),
    (2023, "Nov", "P1", "memo"),
    (2023, "Nov", "P2", "exam"),
    (2023, "Nov", "P2", "memo"),
]

verified_set = set(VERIFIED_DOCUMENTS)

updated = 0
for idx, row in register_df.iterrows():
    key = (
        int(row["year"]) if pd.notna(row["year"]) else None,
        row["exam_session"],
        row["paper"],
        row["document_type"],
    )
    if key in verified_set:
        register_df.loc[idx, "collection_status"] = "verified"
        register_df.loc[idx, "source"] = "DBE"
        register_df.loc[idx, "source_tier"] = 1
        updated += 1

print("Official availability audit completed.")
print(f"Rows updated to verified: {updated}")
print(f"Verified exam/memo targets: {len(VERIFIED_DOCUMENTS)}")

Official availability audit completed.
Rows updated to verified: 12
Verified exam/memo targets: 12


In [14]:
register_df["source_url"] = register_df["source_url"].astype("object")

mask = (
    (register_df["collection_status"] == "verified")
    & (register_df["document_type"].isin(["exam", "memo"]))
    & (register_df["year"].isin([2023, 2024, 2025]))
)

register_df.loc[mask, "source_url"] = register_df.loc[mask, "year"].map(DBE_SOURCE_PAGES)

print("URLs assigned:", mask.sum())

URLs assigned: 12


In [15]:
# ============================================================
# PRIORITY AUDIT VALIDATION
# ============================================================

priority_exam_memos = register_df[
    (register_df["year"].isin([2023, 2024, 2025]))
    & (register_df["exam_session"] == "Nov")
    & (register_df["paper"].isin(["P1", "P2"]))
    & (register_df["document_type"].isin(["exam", "memo"]))
].copy()

print(f"Priority exam/memo documents: {len(priority_exam_memos)}")
print()

display(
    priority_exam_memos[
        [
            "year",
            "exam_session",
            "paper",
            "document_type",
            "source",
            "source_tier",
            "collection_status",
            "source_url",
        ]
    ].sort_values(["year", "paper", "document_type"])
)

assert len(priority_exam_memos) == 12
assert (priority_exam_memos["collection_status"] == "verified").all()
assert (priority_exam_memos["source"] == "DBE").all()
assert (priority_exam_memos["source_tier"] == 1).all()

print("Priority audit validation passed.")

Priority exam/memo documents: 12



,year,exam_session,paper,document_type,source,source_tier,collection_status,source_url
144,2023,Nov,P1,exam,DBE,1,verified,https://www.education.gov.za/2023NSCNovemberpa...
145,2023,Nov,P1,memo,DBE,1,verified,https://www.education.gov.za/2023NSCNovemberpa...
146,2023,Nov,P2,exam,DBE,1,verified,https://www.education.gov.za/2023NSCNovemberpa...
147,2023,Nov,P2,memo,DBE,1,verified,https://www.education.gov.za/2023NSCNovemberpa...
160,2024,Nov,P1,exam,DBE,1,verified,https://www.education.gov.za/2024NSCNovemberpa...
161,2024,Nov,P1,memo,DBE,1,verified,https://www.education.gov.za/2024NSCNovemberpa...
162,2024,Nov,P2,exam,DBE,1,verified,https://www.education.gov.za/2024NSCNovemberpa...
163,2024,Nov,P2,memo,DBE,1,verified,https://www.education.gov.za/2024NSCNovemberpa...
176,2025,Nov,P1,exam,DBE,1,verified,https://www.education.gov.za/Curriculum/Nation...
177,2025,Nov,P1,memo,DBE,1,verified,https://www.education.gov.za/Curriculum/Nation...


Priority audit validation passed.


In [16]:
# ============================================================
# SUPPORTING DOCUMENTS: CAPS + EXAM GUIDELINE
# ============================================================

now = datetime.now(timezone.utc).isoformat()

SUPPORTING_DOCUMENTS = [
    {
        "document_id": "caps_mathematics_gr10_12",
        "year": 2011,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "caps",
        "subject": "Mathematics",
        "language": "English",
        "source": "DBE",
        "source_tier": 1,
        "source_url": (
            "https://www.education.gov.za/Portals/0/CD/"
            "National%20Curriculum%20Statements%20and%20Vocational/"
            "CAPS%20FET%20_%20MATHEMATICS%20_%20GR%2010-12%20_%20Web_1133.pdf"
        ),
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "verified",
        "priority": "high",
        "notes": "Official DBE CAPS Mathematics Grades 10-12",
        "date_added": now,
    },
    {
        "document_id": "grade12_mathematics_exam_guideline_2021",
        "year": 2021,
        "exam_session": "NA",
        "paper": "NA",
        "document_type": "guideline",
        "subject": "Mathematics",
        "language": "English",
        "source": "DBE",
        "source_tier": 1,
        "source_url": (
            "https://www.education.gov.za/Portals/0/CD/"
            "2021%20Exam%20Guidelines/"
            "Mathematics%20GR%2012%20Exam%20Guidelines%202021%20Eng.pdf"
        ),
        "file_name": "",
        "file_path": "",
        "curriculum_regime": "CAPS-Grade12",
        "text_extractable": "unknown",
        "ocr_required": "unknown",
        "collection_status": "verified",
        "priority": "high",
        "notes": "Official DBE Grade 12 Mathematics Examination Guideline (2021)",
        "date_added": now,
    },
]

supporting_df = pd.DataFrame(SUPPORTING_DOCUMENTS, columns=REGISTER_COLUMNS)

# Replace any existing rows with same document_id
register_df = register_df[
    ~register_df["document_id"].isin(supporting_df["document_id"])
].copy()

register_df = pd.concat([register_df, supporting_df], ignore_index=True)
register_df = register_df[REGISTER_COLUMNS]

print("Supporting documents added/updated:")
display(
    supporting_df[
        ["document_id", "document_type", "source", "source_tier", "collection_status", "priority"]
    ]
)

Supporting documents added/updated:


,document_id,document_type,source,source_tier,collection_status,priority
0,caps_mathematics_gr10_12,caps,DBE,1,verified,high
1,grade12_mathematics_exam_guideline_2021,guideline,DBE,1,verified,high


In [17]:
# ============================================================
# SAVE AUDITED REGISTER
# ============================================================

register_path = META_DIR / "exam_document_register.csv"
register_df.to_csv(register_path, index=False)

print("Audited register saved.")
print(f"Path: {register_path}")
print(f"Rows: {len(register_df)}")
print()
print(register_df["collection_status"].value_counts())

Audited register saved.
Path: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv
Rows: 208

collection_status
queued      194
verified     14
Name: count, dtype: int64


In [18]:
# ============================================================
# FINAL PRIORITY AUDIT SUMMARY
# ============================================================

priority_docs = register_df[
    (
        (register_df["year"].isin([2023, 2024, 2025]))
        & (register_df["exam_session"] == "Nov")
        & (register_df["paper"].isin(["P1", "P2"]))
        & (register_df["document_type"].isin(["exam", "memo"]))
    )
    | (register_df["document_type"].isin(["caps", "guideline"]))
].copy()

print("=" * 60)
print("MATRICMATH INTELLIGENCE")
print("OFFICIAL DBE AVAILABILITY AUDIT")
print("=" * 60)
print(f"Priority documents in audit set: {len(priority_docs)}")
print()

display(
    priority_docs[
        ["year", "exam_session", "paper", "document_type", "collection_status", "source_tier"]
    ].sort_values(["document_type", "year", "paper"])
)

print("=" * 60)
print("AUDIT COMPLETE")
print("No PDFs were bulk downloaded.")
print("Diagnostic reports remain unaudited until exact Mathematics reports are confirmed.")
print("Next: Notebook 02 – Document Collection & Extraction")
print("=" * 60)

MATRICMATH INTELLIGENCE
OFFICIAL DBE AVAILABILITY AUDIT
Priority documents in audit set: 16



,year,exam_session,paper,document_type,collection_status,source_tier
204,0,NaN,NaN,caps,queued,1
206,2011,NA,NA,caps,verified,1
144,2023,Nov,P1,exam,verified,1
146,2023,Nov,P2,exam,verified,1
160,2024,Nov,P1,exam,verified,1
162,2024,Nov,P2,exam,verified,1
176,2025,Nov,P1,exam,verified,1
178,2025,Nov,P2,exam,verified,1
205,0,NaN,NaN,guideline,queued,1
207,2021,NA,NA,guideline,verified,1


AUDIT COMPLETE
No PDFs were bulk downloaded.
Diagnostic reports remain unaudited until exact Mathematics reports are confirmed.
Next: Notebook 02 – Document Collection & Extraction


In [19]:
# ============================================================
# CLEAN OBSOLETE SUPPORTING PLACEHOLDERS
# ============================================================

# Keep only the verified supporting docs; drop empty year-0 placeholders
mask_bad_support = (
    (register_df["document_type"].isin(["caps", "guideline"]))
    & (
        (register_df["year"] == 0)
        | (register_df["year"].isna())
        | (register_df["document_id"].isin([
            "caps_mathematics_current",
            "nsc_mathematics_exam_guideline"
        ]))
    )
)

print("Rows to remove:", mask_bad_support.sum())
display(register_df.loc[mask_bad_support, ["document_id", "year", "document_type", "collection_status"]])

register_df = register_df.loc[~mask_bad_support].copy()

# Save cleaned register
register_path = META_DIR / "exam_document_register.csv"
register_df.to_csv(register_path, index=False)

print("Cleaned register saved")
print(f"Total rows now: {len(register_df)}")
print(register_df["collection_status"].value_counts())

Rows to remove: 2


,document_id,year,document_type,collection_status
204,caps_mathematics_current,0,caps,queued
205,nsc_mathematics_exam_guideline,0,guideline,queued


Cleaned register saved
Total rows now: 206
collection_status
queued      192
verified     14
Name: count, dtype: int64


In [20]:
priority_docs = register_df[
    (
        (register_df["year"].isin([2023, 2024, 2025]))
        & (register_df["exam_session"] == "Nov")
        & (register_df["paper"].isin(["P1", "P2"]))
        & (register_df["document_type"].isin(["exam", "memo"]))
    )
    | (register_df["document_type"].isin(["caps", "guideline"]))
].copy()

print("Priority documents:", len(priority_docs))
display(
    priority_docs[
        ["year", "exam_session", "paper", "document_type", "collection_status", "source_url"]
    ].sort_values(["document_type", "year", "paper"])
)

Priority documents: 14


,year,exam_session,paper,document_type,collection_status,source_url
206,2011,NA,NA,caps,verified,https://www.education.gov.za/Portals/0/CD/Nati...
144,2023,Nov,P1,exam,verified,https://www.education.gov.za/2023NSCNovemberpa...
146,2023,Nov,P2,exam,verified,https://www.education.gov.za/2023NSCNovemberpa...
160,2024,Nov,P1,exam,verified,https://www.education.gov.za/2024NSCNovemberpa...
162,2024,Nov,P2,exam,verified,https://www.education.gov.za/2024NSCNovemberpa...
176,2025,Nov,P1,exam,verified,https://www.education.gov.za/Curriculum/Nation...
178,2025,Nov,P2,exam,verified,https://www.education.gov.za/Curriculum/Nation...
207,2021,NA,NA,guideline,verified,https://www.education.gov.za/Portals/0/CD/2021...
145,2023,Nov,P1,memo,verified,https://www.education.gov.za/2023NSCNovemberpa...
147,2023,Nov,P2,memo,verified,https://www.education.gov.za/2023NSCNovemberpa...


In [1]:
# Run in a notebook or: python -c "..." from project root
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
META = ROOT / "data" / "metadata"
META.mkdir(parents=True, exist_ok=True)

register_path = META / "exam_document_register.csv"

new_rows = [
    {
        "document_id": "2022_nov_p1_exam_maths",
        "year": 2022,
        "session": "Nov",
        "paper": "P1",
        "document_type": "exam",
        "language": "Eng",
        "source_url": "https://www.education.gov.za/Portals/0/CD/2022NovExams/Non-Languages%20Nov%202022%20PDF/Mathematics/Mathematics%20P1%20Nov%202022%20Eng.pdf?ver=2023-02-02-114231-000",
        "local_path": "data/raw/exams/2022_nov_p1_exam_maths.pdf",
        "collection_status": "owned",
        "verified": True,
        "priority": "high",
        "curriculum_regime": "CAPS-Grade12",
        "notes": "Verified title page",
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "document_id": "2022_nov_p2_exam_maths",
        "year": 2022,
        "session": "Nov",
        "paper": "P2",
        "document_type": "exam",
        "language": "Eng",
        "source_url": "https://www.education.gov.za/Portals/0/CD/2022NovExams/Non-Languages%20Nov%202022%20PDF/Mathematics/Mathematics%20P2%20Nov%202022%20Eng.pdf?ver=2023-02-08-132128-000",
        "local_path": "data/raw/exams/2022_nov_p2_exam_maths.pdf",
        "collection_status": "owned",
        "verified": True,
        "priority": "high",
        "curriculum_regime": "CAPS-Grade12",
        "notes": "Verified title page",
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "document_id": "2022_nov_p1_memo_maths",
        "year": 2022,
        "session": "Nov",
        "paper": "P1",
        "document_type": "memo",
        "language": "Eng",
        "source_url": "https://www.education.gov.za/LinkClick.aspx?fileticket=gytAGGPYSc8%3d&tabid=3294&portalid=0&mid=10986",
        "local_path": "data/raw/memos/2022_nov_p1_memo_maths.pdf",
        "collection_status": "owned",
        "verified": True,
        "priority": "high",
        "curriculum_regime": "CAPS-Grade12",
        "notes": "Verified title page",
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "document_id": "2022_nov_p2_memo_maths",
        "year": 2022,
        "session": "Nov",
        "paper": "P2",
        "document_type": "memo",
        "language": "Eng",
        "source_url": "https://www.education.gov.za/LinkClick.aspx?fileticket=var8F7VpOmU%3d&tabid=3294&portalid=0&mid=10986",
        "local_path": "data/raw/memos/2022_nov_p2_memo_maths.pdf",
        "collection_status": "owned",
        "verified": True,
        "priority": "high",
        "curriculum_regime": "CAPS-Grade12",
        "notes": "Verified title page",
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "document_id": "2022_maths_diagnostic_part1",
        "year": 2022,
        "session": None,
        "paper": None,
        "document_type": "diagnostic",
        "language": "Eng",
        "source_url": "https://www.education.gov.za/Portals/0/Documents/Reports/Diagnostic%20Reports%202022/",
        "local_path": "data/raw/diagnostic_reports/2022_maths_diagnostic_part1.pdf",
        "collection_status": "owned",
        "verified": True,
        "priority": "high",
        "curriculum_regime": "CAPS-Grade12",
        "notes": "Part 1 Content Subjects; Maths chapter inside",
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
]

new_df = pd.DataFrame(new_rows)

if register_path.exists():
    old = pd.read_csv(register_path)
    # drop any previous 2022 rows for these ids, then append
    old = old[~old["document_id"].isin(new_df["document_id"])]
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(register_path, index=False)
print("Register rows:", len(out))
print(out[out["year"] == 2022][["document_id", "document_type", "collection_status", "verified"]])

Register rows: 207
                          document_id document_type collection_status verified
128       2022_may_june_p1_exam_maths          exam            queued      NaN
129       2022_may_june_p1_memo_maths          memo            queued      NaN
130       2022_may_june_p2_exam_maths          exam            queued      NaN
131       2022_may_june_p2_memo_maths          memo            queued      NaN
132  2022_supplementary_p1_exam_maths          exam            queued      NaN
133  2022_supplementary_p1_memo_maths          memo            queued      NaN
134  2022_supplementary_p2_exam_maths          exam            queued      NaN
135  2022_supplementary_p2_memo_maths          memo            queued      NaN
136      2022_feb_march_p1_exam_maths          exam            queued      NaN
137      2022_feb_march_p1_memo_maths          memo            queued      NaN
138      2022_feb_march_p2_exam_maths          exam            queued      NaN
139      2022_feb_march_p2_memo_m